# IEMOCAP Plan A — Ablation Study

Tests 5 configurations on **fold1 (Session1) + fold5 (Session5)** only — ~20 min total.

| Config | Speaker edges | BCL detach | λ_BCL | Layer order |
|--------|--------------|-----------|-------|-------------|
| A baseline | ✓ | ✗ | 0.5 | inter-first |
| B no_speaker | ✗ | ✗ | 0.5 | inter-first |
| C bcl_fix | ✓ | ✓ | 0.1 | inter-first |
| D no_spk_bcl | ✗ | ✓ | 0.1 | inter-first |
| E all | ✗ | ✓ | 0.1 | intra-first |

Fixed across all: `K_LAYERS=2, HIDDEN=256, DROPOUT=0.5, WD=5e-4, EPOCHS=40, PATIENCE=10`

Results → `Dataset/Processed/IEMOCAP/ablation_results.json`

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score
import json, time, copy, warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 3060


In [2]:
# ============================================================
#  PATHS & FIXED HYPERPARAMETERS
# ============================================================
PROJECT_ROOT = Path('/mnt/Work/ML/Thesis/BMVC/Hopeful')
FEAT_DIR     = PROJECT_ROOT / 'Dataset/Processed/IEMOCAP/features'
RESULTS_DIR  = PROJECT_ROOT / 'Dataset/Processed/IEMOCAP'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IEMOCAP_EMOTIONS = ['ang', 'exc', 'fru', 'hap', 'neu', 'sad']
EMO2IDX          = {e: i for i, e in enumerate(IEMOCAP_EMOTIONS)}
NUM_CLASSES      = 6
D_TEXT, D_AUDIO, D_VIS = 1024, 1024, 1160

# Fixed hyperparams — same for all ablation configs
HIDDEN        = 256
K_LAYERS      = 2
W_PAST        = 4
W_FUTURE      = 1
DROPOUT       = 0.5
CHUNK         = 64
EPOCHS        = 40      # reduced for ablation speed
LR            = 2e-4
WEIGHT_DECAY  = 5e-4
BETA_CB       = 0.9999
GAMMA_FOCAL   = 2.0
PATIENCE      = 10
WARMUP_EPOCHS = 5

SESSIONS      = ['Session1','Session2','Session3','Session4','Session5']
TEST_SESSIONS = ['Session1', 'Session5']   # worst + best from prior full run

print(f'Fixed: K={K_LAYERS}, HIDDEN={HIDDEN}, DROPOUT={DROPOUT}, WD={WEIGHT_DECAY}')
print(f'Testing folds: {TEST_SESSIONS}')

# ============================================================
#  ABLATION CONFIGURATIONS  —  vary one axis at a time
# ============================================================
ABLATION_CONFIGS = [
    {
        'name'             : 'A_baseline',
        'desc'             : 'speaker edges | BCL no-detach | λ=0.5 | inter-first',
        'use_speaker_edges': True,
        'detach_bcl'       : False,
        'lambda_bcl'       : 0.5,
        'intra_first'      : False,
    },
    {
        'name'             : 'B_no_speaker',
        'desc'             : 'no speaker edges | BCL no-detach | λ=0.5 | inter-first',
        'use_speaker_edges': False,
        'detach_bcl'       : False,
        'lambda_bcl'       : 0.5,
        'intra_first'      : False,
    },
    {
        'name'             : 'C_bcl_fix',
        'desc'             : 'speaker edges | BCL detach | λ=0.1 | inter-first',
        'use_speaker_edges': True,
        'detach_bcl'       : True,
        'lambda_bcl'       : 0.1,
        'intra_first'      : False,
    },
    {
        'name'             : 'D_no_spk_bcl',
        'desc'             : 'no speaker | BCL detach | λ=0.1 | inter-first',
        'use_speaker_edges': False,
        'detach_bcl'       : True,
        'lambda_bcl'       : 0.1,
        'intra_first'      : False,
    },
    {
        'name'             : 'E_all',
        'desc'             : 'no speaker | BCL detach | λ=0.1 | intra-first',
        'use_speaker_edges': False,
        'detach_bcl'       : True,
        'lambda_bcl'       : 0.1,
        'intra_first'      : True,
    },
]
print(f'{len(ABLATION_CONFIGS)} configs × {len(TEST_SESSIONS)} folds = {len(ABLATION_CONFIGS)*len(TEST_SESSIONS)} runs')

Fixed: K=2, HIDDEN=256, DROPOUT=0.5, WD=0.0005
Testing folds: ['Session1', 'Session5']
5 configs × 2 folds = 10 runs


In [3]:
# ============================================================
#  Load features (once)
# ============================================================
print('Loading IEMOCAP features...')
feats_text   = torch.load(FEAT_DIR / 'text_roberta_large.pt',          weights_only=False)
feats_audio  = torch.load(FEAT_DIR / 'audio_microsoft_wavlm_large.pt', weights_only=False)
feats_siglip = torch.load(FEAT_DIR / 'video_siglip2_temporal.pt',      weights_only=False)
feats_au     = torch.load(FEAT_DIR / 'video_openface_au.pt',           weights_only=False)

visual_absent_map = {
    uid: (float(np.asarray(feats_au[uid]).sum()) == 0.0 and
          float(np.asarray(feats_siglip[uid]).sum()) == 0.0)
    for uid in feats_text
}

labels_df = pd.read_csv(PROJECT_ROOT / 'Dataset/Processed/IEMOCAP/labels.csv')
labels_df = labels_df[labels_df['emotion'].isin(IEMOCAP_EMOTIONS)].copy().reset_index(drop=True)
labels_df['label'] = labels_df['emotion'].map(EMO2IDX)
print(f'Loaded: {len(labels_df)} utterances')
print(labels_df['emotion'].value_counts())

Loading IEMOCAP features...
Loaded: 7380 utterances
emotion
fru    1849
neu    1708
ang    1103
sad    1084
exc    1041
hap     595
Name: count, dtype: int64


In [4]:
# ============================================================
#  Incidence matrix — two variants (with / without speaker edges)
# ============================================================
def build_H(N, speakers, use_speaker_edges, w_past=W_PAST, w_future=W_FUTURE):
    offsets = [0, N, 2*N, 3*N, 4*N]
    rows, cols = [], []

    if use_speaker_edges:
        unique_spk = list(dict.fromkeys(speakers))
        K = len(unique_spk); spk2idx = {s: i for i, s in enumerate(unique_spk)}
        E = 7*N + 5*K
    else:
        E = 7*N

    for i in range(N):
        for node in [i, N+i, 2*N+i, 3*N+i, 4*N+i]:       # MM
            rows.append(node); cols.append(i)
        for node in [2*N+i, 3*N+i, 4*N+i]:                # VA
            rows.append(node); cols.append(N+i)
        window = range(max(0, i-w_past), min(N, i+w_future+1))
        for m, off in enumerate(offsets):                  # Contextual
            for j in window:
                rows.append(off+j); cols.append(2*N + m*N + i)
        if use_speaker_edges:                              # Speaker
            k = spk2idx[speakers[i]]
            for m, off in enumerate(offsets):
                rows.append(off+i); cols.append(7*N + k*5 + m)

    V = 5*N
    H = torch.sparse_coo_tensor(
        torch.tensor([rows, cols]), torch.ones(len(rows)), (V, E)
    ).coalesce().to_dense()

    et_parts = [
        torch.zeros(N, dtype=torch.long),
        torch.ones(N,  dtype=torch.long),
        torch.full((5*N,), 2, dtype=torch.long),
    ]
    if use_speaker_edges:
        et_parts.append(torch.full((5*K,), 3, dtype=torch.long))
    return H, torch.cat(et_parts)


# Quick check
for use_spk in [True, False]:
    H_v, et_v = build_H(10, ['F','M']*5, use_spk)
    print(f'spk={use_spk}: H={H_v.shape}  et={et_v.bincount().tolist()}')

spk=True: H=torch.Size([50, 80])  et=[10, 10, 50, 10]
spk=False: H=torch.Size([50, 70])  et=[10, 10, 50]


In [5]:
# ============================================================
#  Dialog builder
# ============================================================
def build_dialog(dialog_id, dialog_df, use_speaker_edges):
    sub  = dialog_df[dialog_df['dialog'] == dialog_id].sort_values('start_time').reset_index(drop=True)
    utts = sub['utt_id'].tolist()
    N    = len(utts)
    text_feat  = torch.from_numpy(np.stack([np.asarray(feats_text[u],  dtype=np.float32) for u in utts]))
    audio_feat = torch.from_numpy(np.stack([np.asarray(feats_audio[u], dtype=np.float32) for u in utts]))
    vis_list   = [np.concatenate([np.asarray(feats_au[u], dtype=np.float32),
                                  np.asarray(feats_siglip[u], dtype=np.float32)], axis=-1) for u in utts]
    vis_feat   = torch.from_numpy(np.stack(vis_list))
    vis_absent = torch.tensor([visual_absent_map[u] for u in utts], dtype=torch.bool)
    labels_t   = torch.tensor(sub['label'].tolist(), dtype=torch.long)
    speakers   = sub['speaker'].tolist()
    H_mat, et  = build_H(N, speakers, use_speaker_edges)
    return dict(text=text_feat, audio=audio_feat, vis=vis_feat, vis_absent=vis_absent,
                labels=labels_t, speakers=speakers, N=N, H_mat=H_mat, et=et)


def build_session_dialogs(sess, use_speaker_edges):
    sess_df = labels_df[labels_df['session'] == sess]
    return [build_dialog(d, sess_df, use_speaker_edges)
            for d in sess_df['dialog'].unique()
            if len(sess_df[sess_df['dialog']==d]) >= 2]


print('Pre-building dialog cache for all sessions × both incidence variants...')
t0 = time.time()
dialog_cache = {}
for sess in SESSIONS:
    for use_spk in [True, False]:
        dialog_cache[(sess, use_spk)] = build_session_dialogs(sess, use_spk)
print(f'Done in {time.time()-t0:.1f}s')
print(f'Example — Session1 spk=True: {len(dialog_cache[("Session1",True)])} dialogs')

Pre-building dialog cache for all sessions × both incidence variants...
Done in 0.6s
Example — Session1 spk=True: 28 dialogs


In [6]:
# ============================================================
#  Model (parameterised for all ablation variants)
# ============================================================
class ModalityProjector(nn.Module):
    def __init__(self, in_dim, out_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim), nn.GELU(), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)


class VisualProjector(nn.Module):
    def __init__(self, vis_in, d, dropout):
        super().__init__()
        self.proj = ModalityProjector(vis_in, d, dropout)
        self.visual_absent_embed = nn.Parameter(torch.zeros(3, d))
    def forward(self, vis, vis_absent):
        h = self.proj(vis)
        if vis_absent.any():
            h = torch.where(vis_absent.view(-1,1,1).expand_as(h),
                            self.visual_absent_embed.unsqueeze(0).expand_as(h), h)
        return h


class HypergraphConvLayer(nn.Module):
    def __init__(self, d, num_edge_types, dropout, chunk):
        super().__init__()
        self.chunk = chunk
        self.node_attn     = nn.Linear(d, 1)
        self.edge_type_emb = nn.Embedding(num_edge_types, d)
        self.hedge_attn    = nn.Linear(2*d, 1)
        self.W_update      = nn.Linear(d, d)
        self.norm          = nn.LayerNorm(d)
        self.dropout       = nn.Dropout(dropout)

    def forward(self, X, H, et):
        V, E = X.shape[0], H.shape[1]
        ns = self.node_attn(X).expand(-1, E)
        ns = ns.masked_fill(H == 0, -1e9)
        B  = (F.softmax(ns, dim=0) * H).T @ X
        Bt = B + self.edge_type_emb(et)
        hat = torch.zeros(V, E, device=X.device)
        for s in range(0, V, self.chunk):
            e = min(s + self.chunk, V)
            inp = torch.cat([
                X[s:e].unsqueeze(1).expand(-1, E, -1),
                Bt.unsqueeze(0).expand(e-s, -1, -1)
            ], dim=-1)
            sc = self.hedge_attn(inp).squeeze(-1)
            sc = sc.masked_fill(H[s:e] == 0, -1e9)
            hat[s:e] = F.softmax(sc, dim=1) * H[s:e]
        return self.norm(X + self.dropout(F.gelu(self.W_update(hat @ B))))


class PlanAModel(nn.Module):
    def __init__(self, d, num_classes, k_layers, dropout, num_edge_types,
                 detach_bcl, intra_first, chunk=CHUNK):
        super().__init__()
        self.k_layers    = k_layers
        self.detach_bcl  = detach_bcl
        self.intra_first = intra_first
        self.text_proj   = ModalityProjector(D_TEXT,  d, dropout)
        self.audio_proj  = ModalityProjector(D_AUDIO, d, dropout)
        self.vis_proj    = VisualProjector(D_VIS, d, dropout)
        self.layers      = nn.ModuleList([
            HypergraphConvLayer(d, num_edge_types, dropout, chunk)
            for _ in range(k_layers)
        ])
        self.arc_attn   = nn.Linear(d, 1)
        self.fusion_mlp = nn.Sequential(
            nn.Linear(3*d, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d, num_classes))
        self.bcl_head = nn.Sequential(nn.Linear(3*d, 256), nn.GELU(), nn.Linear(256, 128))

    def forward(self, text, audio, vis, vis_absent, H_mat, edge_types, return_feats=False):
        N = text.shape[0]
        ht = self.text_proj(text)
        ha = self.audio_proj(audio)
        hv = self.vis_proj(vis, vis_absent)
        X  = torch.cat([ht, ha, hv[:,0], hv[:,1], hv[:,2]], dim=0)
        H  = H_mat.to(X.device); ET = edge_types.to(X.device)

        intra_mask = (ET == 1) | (ET == 2)
        inter_mask = (ET == 0) | (ET == 3)
        H_intra = H[:, intra_mask]; ET_intra = ET[intra_mask]
        H_inter = H[:, inter_mask]; ET_inter = ET[inter_mask]

        for li, layer in enumerate(self.layers):
            if self.intra_first:
                X = layer(X, H_intra, ET_intra) if li % 2 == 0 else layer(X, H_inter, ET_inter)
            else:
                X = layer(X, H_inter, ET_inter) if li % 2 == 0 else layer(X, H_intra, ET_intra)

        ht_out = X[0*N:1*N]; ha_out = X[1*N:2*N]
        vis_stack  = torch.stack([X[2*N:3*N], X[3*N:4*N], X[4*N:5*N]], dim=1)
        arc_w      = F.softmax(self.arc_attn(vis_stack).squeeze(-1), dim=-1)
        hv_pooled  = (arc_w.unsqueeze(-1) * vis_stack).sum(1)
        h_fused    = torch.cat([ht_out, ha_out, hv_pooled], dim=-1)
        logits     = self.fusion_mlp(h_fused)

        if return_feats:
            inp   = h_fused.detach() if self.detach_bcl else h_fused
            feats = F.normalize(self.bcl_head(inp), dim=-1)
            return logits, feats, arc_w
        return logits


# Quick sanity check
_N, _sp = 8, ['F','M']*4
_H, _et = build_H(_N, _sp, True)
_m = PlanAModel(32, NUM_CLASSES, 2, 0.3, 4, False, False, chunk=16).to(DEVICE)
_lg, _ft, _aw = _m(torch.randn(_N,1024).to(DEVICE), torch.randn(_N,1024).to(DEVICE),
                   torch.randn(_N,3,1160).to(DEVICE), torch.zeros(_N,dtype=torch.bool).to(DEVICE),
                   _H, _et, return_feats=True)
_lg.sum().backward()
print(f'Sanity OK — logits={_lg.shape}  feats={_ft.shape}  arc={_aw.shape}')
del _m, _lg, _ft, _aw, _H, _et
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

Sanity OK — logits=torch.Size([8, 6])  feats=torch.Size([8, 128])  arc=torch.Size([8, 3])


In [7]:
# ============================================================
#  Loss functions
# ============================================================
class CBFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.9999, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        nc = torch.tensor(class_counts, dtype=torch.float32)
        eff = (1 - beta**nc) / (1 - beta)
        w = (1 - beta) / eff; w = w / w.sum() * len(w)
        self.register_buffer('weights', w)
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weights, reduction='none')
        return ((1 - torch.exp(-ce))**self.gamma * ce).mean()


class BalancedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.T = temperature
    def forward(self, feats, labels):
        N = feats.shape[0]
        if N < 4 or labels.unique().numel() < 2:
            return torch.tensor(0.0, device=feats.device, requires_grad=True)
        sim = feats @ feats.T / self.T
        diag = torch.eye(N, dtype=torch.bool, device=feats.device)
        pos  = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~diag
        n_p  = pos.sum(1, keepdim=True).float().clamp(min=1)
        ld   = torch.logsumexp(sim.masked_fill(diag, -1e9), dim=1, keepdim=True)
        return -(pos.float() * (sim - ld) / n_p).sum(1).mean()


print('Loss functions defined.')

Loss functions defined.


In [8]:
# ============================================================
#  Training utilities
# ============================================================
def compute_wf1(labels, preds):
    return float(f1_score(labels, preds, average='weighted', zero_division=0))


def evaluate(model, dialogs, crit_focal, crit_bcl, lambda_bcl):
    model.eval()
    total_loss, preds, true = 0.0, [], []
    with torch.no_grad():
        for d in dialogs:
            if d['N'] < 2: continue
            logits, feats, _ = model(
                d['text'].to(DEVICE), d['audio'].to(DEVICE),
                d['vis'].to(DEVICE),  d['vis_absent'].to(DEVICE),
                d['H_mat'], d['et'], return_feats=True)
            labels = d['labels'].to(DEVICE)
            total_loss += (crit_focal(logits, labels) + lambda_bcl * crit_bcl(feats, labels)).item()
            preds.extend(logits.argmax(-1).cpu().tolist())
            true.extend(labels.cpu().tolist())
    return total_loss / max(len(dialogs),1), compute_wf1(true, preds)


def train_one_fold(train_dialogs, val_dialogs, cfg):
    num_et = 4 if cfg['use_speaker_edges'] else 3
    model  = PlanAModel(
        d=HIDDEN, num_classes=NUM_CLASSES, k_layers=K_LAYERS, dropout=DROPOUT,
        num_edge_types=num_et, detach_bcl=cfg['detach_bcl'], intra_first=cfg['intra_first']
    ).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    all_tl = torch.cat([d['labels'] for d in train_dialogs])
    cc = [(all_tl == c).sum().item() for c in range(NUM_CLASSES)]
    crit_focal = CBFocalLoss(cc, BETA_CB, GAMMA_FOCAL).to(DEVICE)
    crit_bcl   = BalancedContrastiveLoss(0.07).to(DEVICE)
    lam        = cfg['lambda_bcl']

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, EPOCHS-WARMUP_EPOCHS), eta_min=1e-6)

    hist = {'train_loss':[], 'val_loss':[], 'train_wf1':[], 'val_wf1':[]}
    best_wf1, best_state, no_improve = 0.0, None, 0
    t0 = time.time()

    for epoch in range(EPOCHS):
        if epoch < WARMUP_EPOCHS:
            for pg in optimizer.param_groups: pg['lr'] = LR * (epoch+1) / WARMUP_EPOCHS

        model.train()
        tr_loss, preds, true = 0.0, [], []
        order = list(range(len(train_dialogs))); np.random.shuffle(order)
        for di in order:
            d = train_dialogs[di]
            if d['N'] < 2: continue
            optimizer.zero_grad()
            logits, feats, _ = model(
                d['text'].to(DEVICE), d['audio'].to(DEVICE),
                d['vis'].to(DEVICE),  d['vis_absent'].to(DEVICE),
                d['H_mat'], d['et'], return_feats=True)
            labels = d['labels'].to(DEVICE)
            loss = crit_focal(logits, labels) + lam * crit_bcl(feats, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss += loss.item()
            preds.extend(logits.argmax(-1).detach().cpu().tolist())
            true.extend(labels.cpu().tolist())

        if epoch >= WARMUP_EPOCHS: scheduler.step()

        val_loss, val_wf1  = evaluate(model, val_dialogs, crit_focal, crit_bcl, lam)
        train_wf1 = compute_wf1(true, preds)
        hist['train_loss'].append(tr_loss / max(len(train_dialogs),1))
        hist['val_loss'].append(val_loss)
        hist['train_wf1'].append(train_wf1)
        hist['val_wf1'].append(val_wf1)

        if val_wf1 > best_wf1:
            best_wf1  = val_wf1
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'    early stop ep{epoch+1}  best={best_wf1:.4f}')
                break

        if (epoch+1) % 10 == 0:
            print(f'    ep{epoch+1:3d}: tr={train_wf1:.4f} val={val_wf1:.4f} '
                  f'vloss={val_loss:.4f} [{(time.time()-t0)/60:.1f}min]')

    best_ep = int(np.argmax(hist['val_wf1']))
    gap     = hist['train_wf1'][best_ep] - hist['val_wf1'][best_ep]
    print(f'    best_ep={best_ep+1}  val_wf1={best_wf1:.4f}  '
          f'train_wf1@best={hist["train_wf1"][best_ep]:.4f}  gap={gap:.4f}  '
          f'[{(time.time()-t0)/60:.1f}min]')
    return best_wf1, hist, n_params


print('Training utilities defined.')

Training utilities defined.


In [9]:
# ============================================================
#  Run ablation — 5 configs × 2 folds
# ============================================================
all_results  = {}
summary_rows = []

for cfg in ABLATION_CONFIGS:
    print(f'\n{"="*65}')
    print(f'Config: {cfg["name"]}')
    print(f'  {cfg["desc"]}')
    cfg_results = {}

    for test_sess in TEST_SESSIONS:
        print(f'\n  -- Fold: test={test_sess} --')
        use_spk       = cfg['use_speaker_edges']
        val_dialogs   = dialog_cache[(test_sess, use_spk)]
        train_dialogs = [d for s in SESSIONS if s != test_sess
                           for d in dialog_cache[(s, use_spk)]]

        torch.manual_seed(42); np.random.seed(42)
        best_wf1, hist, n_params = train_one_fold(train_dialogs, val_dialogs, cfg)

        best_ep = int(np.argmax(hist['val_wf1']))
        cfg_results[test_sess] = {
            'best_val_wf1'      : float(best_wf1),
            'best_epoch'        : best_ep + 1,
            'train_wf1_at_best' : float(hist['train_wf1'][best_ep]),
            'gap_at_best'       : float(hist['train_wf1'][best_ep] - hist['val_wf1'][best_ep]),
            'val_loss_at_best'  : float(hist['val_loss'][best_ep]),
            'n_epochs_run'      : len(hist['val_wf1']),
            'val_wf1_curve'     : hist['val_wf1'],
            'train_wf1_curve'   : hist['train_wf1'],
            'val_loss_curve'    : hist['val_loss'],
            'n_params'          : n_params,
        }

    wf1s = [cfg_results[s]['best_val_wf1'] for s in TEST_SESSIONS]
    gaps = [cfg_results[s]['gap_at_best']  for s in TEST_SESSIONS]
    row  = {
        'config'   : cfg['name'],
        'desc'     : cfg['desc'],
        'mean_wf1' : float(np.mean(wf1s)),
        'S1_wf1'   : float(wf1s[0]),
        'S5_wf1'   : float(wf1s[1]),
        'mean_gap' : float(np.mean(gaps)),
        'n_params' : cfg_results[TEST_SESSIONS[0]]['n_params'],
    }
    summary_rows.append(row)
    all_results[cfg['name']] = {'config': cfg, 'folds': cfg_results, 'summary': row}
    print(f'\n  >> mean_wf1={row["mean_wf1"]:.4f}  S1={row["S1_wf1"]:.4f}  '
          f'S5={row["S5_wf1"]:.4f}  gap={row["mean_gap"]:.4f}')

print(f'\n{"="*65}')
print('All configs done.')


Config: A_baseline
  speaker edges | BCL no-detach | λ=0.5 | inter-first

  -- Fold: test=Session1 --
    ep 10: tr=0.6229 val=0.4921 vloss=2.3474 [0.2min]
    ep 20: tr=0.7557 val=0.5154 vloss=2.4477 [0.4min]
    early stop ep23  best=0.5785
    best_ep=13  val_wf1=0.5785  train_wf1@best=0.6685  gap=0.0900  [0.4min]

  -- Fold: test=Session5 --
    ep 10: tr=0.6165 val=0.5816 vloss=2.3538 [0.2min]
    ep 20: tr=0.7454 val=0.6365 vloss=2.3926 [0.4min]
    ep 30: tr=0.8177 val=0.6200 vloss=2.4883 [0.5min]
    early stop ep35  best=0.6485
    best_ep=25  val_wf1=0.6485  train_wf1@best=0.7830  gap=0.1345  [0.6min]

  >> mean_wf1=0.6135  S1=0.5785  S5=0.6485  gap=0.1122

Config: B_no_speaker
  no speaker edges | BCL no-detach | λ=0.5 | inter-first

  -- Fold: test=Session1 --
    ep 10: tr=0.5956 val=0.4503 vloss=2.4562 [0.2min]
    ep 20: tr=0.7289 val=0.4906 vloss=2.5811 [0.4min]
    ep 30: tr=0.8009 val=0.5407 vloss=2.5374 [0.5min]
    early stop ep38  best=0.5667
    best_ep=28  val_w

In [10]:
# ============================================================
#  Results table + save
# ============================================================
print('ABLATION SUMMARY  (fold1=Session1 + fold5=Session5, IEMOCAP)')
print(f'{"Config":<22} {"S1 WF1":>8} {"S5 WF1":>8} {"Mean WF1":>10} {"Gap":>8} {"Params":>10}')
print('-'*70)
for row in summary_rows:
    print(f'{row["config"]:<22} {row["S1_wf1"]:>8.4f} {row["S5_wf1"]:>8.4f} '
          f'{row["mean_wf1"]:>10.4f} {row["mean_gap"]:>8.4f} {row["n_params"]:>10,}')

best_cfg = max(summary_rows, key=lambda r: r['mean_wf1'])
print(f'\nWinner: {best_cfg["config"]}  (mean_wf1={best_cfg["mean_wf1"]:.4f})')

out_path = RESULTS_DIR / 'ablation_results.json'
with open(out_path, 'w') as f:
    json.dump({'summary': summary_rows, 'configs': all_results}, f, indent=2)
print(f'Saved: {out_path}')

ABLATION SUMMARY  (fold1=Session1 + fold5=Session5, IEMOCAP)
Config                   S1 WF1   S5 WF1   Mean WF1      Gap     Params
----------------------------------------------------------------------
A_baseline               0.5785   0.6485     0.6135   0.1122  1,389,451
B_no_speaker             0.5667   0.6353     0.6010   0.2066  1,388,939
C_bcl_fix                0.5730   0.6430     0.6080   0.1199  1,389,451
D_no_spk_bcl             0.5748   0.6309     0.6029   0.1465  1,388,939
E_all                    0.5743   0.6377     0.6060   0.1513  1,388,939

Winner: A_baseline  (mean_wf1=0.6135)
Saved: /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/IEMOCAP/ablation_results.json


In [11]:
# ============================================================
#  Visualise: WF1 curves per config per fold
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue','darkorange','forestgreen','crimson','purple']

for ax, test_sess in zip(axes, TEST_SESSIONS):
    for ci, cfg in enumerate(ABLATION_CONFIGS):
        curve = all_results[cfg['name']]['folds'][test_sess]['val_wf1_curve']
        ep    = range(1, len(curve)+1)
        ax.plot(ep, curve, color=colors[ci], lw=1.8, label=cfg['name'])
    ax.set_title(f'Val WF1 — {test_sess}', fontsize=11)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Val WF1')
    ax.set_ylim(0, 0.8); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('IEMOCAP Ablation — Validation WF1 Curves', fontsize=12)
plt.tight_layout()
viz_path = RESULTS_DIR / 'ablation_wf1_curves.png'
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {viz_path}')

Saved: /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/IEMOCAP/ablation_wf1_curves.png
